## Dublin Mobility Project analysis
* This Notebook's objective is to extract, clean and analyse cycling, walking and transit infrastructure in the Dundrum-Sandyford area
* In the first two cells, i installed the essential geospatial libraries for the project & setups, imports and global paths

In [1]:
!pip install osmnx geopandas pandas shapely matplotlib

import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import box
from pathlib import Path

# Setup global directory path
processed_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "processed"
print("✅ Environment successfully configured.")


✅ Environment successfully configured.


In [2]:
# Cell 1: Setup, Imports, and Global Paths

import pandas as pd
import geopandas as gpd
import osmnx as ox
from pathlib import Path

# Define global directory paths
base_dir = Path.home() / "Desktop" / "Dublin_Mobility_Project"
raw_dir = base_dir / "raw_data"
processed_dir = base_dir / "processed_data"

# Create directories if they don't exist
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete. Directories ready.")

Setup complete. Directories ready.


## Phase 2: Data Acquisition
*Downloads OpenstreetNetwork data

In [3]:
# Cell 2: Phase 1 - Downloading Street Network

# Configure Logging
try:
    ox.settings.log_console = True
    ox.settings.use_cache = True
except AttributeError:
    ox.config(log_console=True, use_cache=True)

# Define Bounding Box (Dundrum-Sandyford Corridor)
bbox = {
    'north': 53.300,
    'south': 53.250,
    'east': -6.180,
    'west': -6.270
}

print("Fetching street network from OpenStreetMap. This may take 1-2 minutes...")

# Download Data (Handles OSMnx version differences)
try:
    # OSMnx v2.0+ format: (west, south, east, north)
    G = ox.graph_from_bbox(
        bbox=(bbox['west'], bbox['south'], bbox['east'], bbox['north']), 
        network_type="all"
    )
except TypeError:
    # Legacy OSMnx (< v2.0) format: (north, south, east, west)
    G = ox.graph_from_bbox(
        bbox['north'], bbox['south'], bbox['east'], bbox['west'], 
        network_type="all"
    )

print("Download complete. Converting to GeoDataFrame...")
nodes, edges = ox.graph_to_gdfs(G)

# Clean Data for GeoJSON Export (Avoids the Geometry AttributeError)
print("Cleaning list attributes to prepare for GeoJSON export...")
edges_clean = edges.copy()

for col in edges_clean.columns:
    if col != "geometry":  # Safely skip the spatial geometry column
        edges_clean[col] = edges_clean[col].apply(
            lambda x: str(x) if isinstance(x, list) else x
        )

# Save File (Uses raw_dir defined in Cell 1)
edges_file = raw_dir / "dublin_street_edges.geojson"
edges_clean.to_file(edges_file, driver="GeoJSON")
print(f"SUCCESS! Street edges saved to:\n{edges_file}")

Fetching street network from OpenStreetMap. This may take 1-2 minutes...
Download complete. Converting to GeoDataFrame...
Cleaning list attributes to prepare for GeoJSON export...
SUCCESS! Street edges saved to:
C:\Users\My PC\Desktop\Dublin_Mobility_Project\raw_data\dublin_street_edges.geojson


# Phase 2: Processing Active Travel and cycle Network
* Acquire census data
* NTA Luas stop Locations and GTFs Schedule
* Pedestrian cycling road networks/cycling infrastructure


# Dublin cycle networks data/infrastructure

In [4]:
# 1. Define folder paths
active_file = raw_dir / "dublin_active_travel.geojson"
edges_file = raw_dir / "dublin_street_edges.geojson"
print("Loading raw datasets...")
active_gdf = gpd.read_file(active_file)
edges_gdf = gpd.read_file(edges_file)

# 2. Filter dedicated off-road cycleways and tracks from active travel data
offroad_cycle = active_gdf[
    (active_gdf.get('highway') == 'cycleway') | 
    (active_gdf.get('bicycle') == 'designated')
].copy()

# 3. Robustly filter on-street cycle lanes from main road network
cycle_cols = [col for col in ['cycleway', 'cycleway:left', 'cycleway:right', 'cycleway:both', 'bicycle'] if col in edges_gdf.columns]

# Initialize mask as a boolean Series matching the GeoDataFrame index length
onstreet_mask = pd.Series(False, index=edges_gdf.index)

for col in cycle_cols:
    onstreet_mask = onstreet_mask | (
        edges_gdf[col].notna() & 
        (~edges_gdf[col].astype(str).str.lower().isin(['no', 'none', 'nan', 'false']))
    )

onstreet_cycle = edges_gdf[onstreet_mask].copy()

# 4. Combine off-road cycle paths and on-street cycle lanes
cycle_network = gpd.GeoDataFrame(
    pd.concat([offroad_cycle, onstreet_cycle], ignore_index=True),
    crs=edges_gdf.crs
)

# 5. Clean list/dict attributes for GeoJSON export compatibility
for col in cycle_network.columns:
    if cycle_network[col].apply(lambda x: isinstance(x, (list, dict))).any():
        cycle_network[col] = cycle_network[col].astype(str)

# 6. Save isolated cycle network to processed folder
output_file = processed_dir / "dublin_cycle_network.geojson"
cycle_network.to_file(output_file, driver="GeoJSON")

print("\n--- Cycling Infrastructure Summary ---")
print(f"• Off-road cycle paths/tracks: {len(offroad_cycle)}")
print(f"• On-street roads with cycle lanes: {len(onstreet_cycle)}")
print(f"• Total cycling network segments: {len(cycle_network)}")
print(f"• Saved cleanly to: {output_file}")

Loading raw datasets...


C:\Users\My PC\anaconda3\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)



--- Cycling Infrastructure Summary ---
• Off-road cycle paths/tracks: 649
• On-street roads with cycle lanes: 0
• Total cycling network segments: 649
• Saved cleanly to: C:\Users\My PC\Desktop\Dublin_Mobility_Project\processed_data\dublin_cycle_network.geojson


# Downloading the Luas stops within that corridor under study

In [5]:
import time
# 1. Setup Desktop project directory paths

output_file = processed_dir / "dublin_luas_stops.geojson"

# 2. Dundrum - Sandyford Bounding Box
west, south, east, north = -6.270, 53.250, -6.180, 53.300

# Targeted, lightweight Luas tags
luas_tags = {
    'railway': ['tram_stop', 'station'],
    'station': 'light_rail'
}

# Reliable Overpass API endpoints
overpass_endpoints = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]

ox.settings.overpass_rate_limit = False
ox.settings.requests_timeout = 60

luas_gdf = None
print("Fetching Luas Green Line stops...")

# Attempt fast download across mirrors
for url in overpass_endpoints:
    print(f"Connecting to: {url} ...")
    ox.settings.overpass_url = url
    try:
        try:
            raw_features = ox.features_from_bbox(bbox=(west, south, east, north), tags=luas_tags)
        except TypeError:
            raw_features = ox.features_from_bbox(north, south, east, west, tags=luas_tags)
            
        if len(raw_features) > 0:
            luas_gdf = raw_features.copy()
            print(f"Successfully fetched data from {url}!")
            break
    except Exception as err:
        print(f"Server busy ({err}). Trying next endpoint...")
        time.sleep(1)

# 3. Fallback: If servers are unreachable, use verified Dundrum-Sandyford Green Line station coordinates
if luas_gdf is None or len(luas_gdf) == 0:
    print("\nOverpass servers unavailable. Using verified Luas Green Line station dataset directly...")
    
    luas_data = [
        {"name": "Dundrum Luas Stop", "latitude": 53.2922, "longitude": -6.2447, "line": "Green Line"},
        {"name": "Balally Luas Stop", "latitude": 53.2861, "longitude": -6.2367, "line": "Green Line"},
        {"name": "Kilmacud Luas Stop", "latitude": 53.2831, "longitude": -6.2120, "line": "Green Line"},
        {"name": "Stillorgan Luas Stop", "latitude": 53.2789, "longitude": -6.2086, "line": "Green Line"},
        {"name": "Sandyford Luas Stop", "latitude": 53.2778, "longitude": -6.2044, "line": "Green Line"},
        {"name": "Central Park Luas Stop", "latitude": 53.2700, "longitude": -6.2036, "line": "Green Line"},
        {"name": "Glencairn Luas Stop", "latitude": 53.2683, "longitude": -6.2078, "line": "Green Line"},
        {"name": "The Gallops Luas Stop", "latitude": 53.2631, "longitude": -6.2081, "line": "Green Line"},
        {"name": "Leopardstown Valley Luas Stop", "latitude": 53.2594, "longitude": -6.1989, "line": "Green Line"}
    ]
    
    df = pd.DataFrame(luas_data)
    geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
    luas_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 4. Convert polygon station boundaries to point centroids
luas_gdf['geometry'] = luas_gdf['geometry'].centroid

# Clean list/dict attributes for GeoJSON export
for col in luas_gdf.columns:
    if luas_gdf[col].apply(lambda x: isinstance(x, (list, dict))).any():
        luas_gdf[col] = luas_gdf[col].astype(str)

# Deduplicate by stop name if present
if 'name' in luas_gdf.columns:
    luas_gdf = luas_gdf.drop_duplicates(subset=['name']).copy()

# 5. Save output file
luas_gdf.to_file(output_file, driver="GeoJSON")

print("\n--- Luas Transit Hub Summary ---")
print(f"• Total Luas stops isolated: {len(luas_gdf)}")
if 'name' in luas_gdf.columns:
    print("• Stations included:")
    for name in sorted(luas_gdf['name'].dropna().unique()):
        print(f"  - {name}")

print(f"\nSaved cleanly to: {output_file}")

Fetching Luas Green Line stops...
Connecting to: https://overpass.kumi.systems/api/interpreter ...
Successfully fetched data from https://overpass.kumi.systems/api/interpreter!

--- Luas Transit Hub Summary ---
• Total Luas stops isolated: 10
• Stations included:
  - Balally
  - Ballyogan Wood
  - Central Park
  - Dundrum
  - Glencairn
  - Kilmacud
  - Leopardstown Valley
  - Sandyford
  - Stillorgan
  - The Gallops

Saved cleanly to: C:\Users\My PC\Desktop\Dublin_Mobility_Project\processed_data\dublin_luas_stops.geojson


C:\Users\My PC\AppData\Local\Temp\ipykernel_17372\1347219830.py:68: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  luas_gdf['geometry'] = luas_gdf['geometry'].centroid


# Processing Census boundaries

In [6]:
print("Loading raw files...")
sa_gdf = gpd.read_file(raw_dir / "cso_small_area_boundaries.geojson")
saps_df = pd.read_csv(raw_dir / "cso_saps_data.csv", low_memory=False)

# 1. Helper function to clean codes and strip common prefixes
def clean_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace("SA2022_", "", regex=False)
        .str.replace("SA_", "", regex=False)
    )

# 2. Identify potential candidate key columns
boundary_candidates = [c for c in sa_gdf.columns if any(k in c.upper() for k in ['SA', 'GEOG', 'GUID', 'CODE', 'ID'])]
saps_candidates = [c for c in saps_df.columns if any(k in c.upper() for k in ['SA', 'GEOG', 'GUID', 'CODE', 'ID'])]

print(f"Scanning candidate boundary keys: {boundary_candidates}")
print(f"Scanning candidate SAPS CSV keys: {saps_candidates[:6]}")

# 3. Test all candidate pairs to find maximum overlap
best_pair = None
max_matches = 0

for b_col in boundary_candidates:
    b_series = clean_id(sa_gdf[b_col])
    for s_col in saps_candidates:
        s_series = clean_id(saps_df[s_col])
        matches = len(set(b_series).intersection(set(s_series)))
        if matches > max_matches:
            max_matches = matches
            best_pair = (b_col, s_col)

# 4. Perform Join if valid match is found
if best_pair and max_matches > 0:
    b_key, s_key = best_pair
    print(f"\nSuccess! Optimal match found:")
    print(f"• Boundary Key: '{b_key}' <---> SAPS Key: '{s_key}'")
    print(f"• Matching Small Area IDs: {max_matches}")
    
    sa_gdf['join_key'] = clean_id(sa_gdf[b_key])
    saps_df['join_key'] = clean_id(saps_df[s_key])
    
    demographic_gdf = sa_gdf.merge(saps_df, on='join_key', how='inner')
    
    # 5. Reproject to EPSG:2157 (Irish Transverse Mercator)
    print("\nReprojecting to EPSG:2157 (ITM meters)...")
    demographic_itm = demographic_gdf.to_crs(epsg=2157)
    
    # 6. Clip to Dundrum - Sandyford Study Area Bounds
    study_bbox_wgs84 = box(-6.270, 53.250, -6.180, 53.300)
    bbox_itm = gpd.GeoSeries([study_bbox_wgs84], crs="EPSG:4326").to_crs(epsg=2157).geometry.iloc[0]
    
    dundrum_demographics = demographic_itm[demographic_itm.intersects(bbox_itm)].copy()
    
    # Clean complex columns before GeoJSON export
    for col in dundrum_demographics.columns:
        if dundrum_demographics[col].apply(lambda x: isinstance(x, (list, dict))).any():
            dundrum_demographics[col] = dundrum_demographics[col].astype(str)
            
    output_file = processed_dir / "dublin_demographics_sa2022.geojson"
    dundrum_demographics.to_crs(epsg=4326).to_file(output_file, driver="GeoJSON")
    
    print("\n--- Demand Layer Processing Complete ---")
    print(f"• Total Small Areas in Dundrum-Sandyford study area: {len(dundrum_demographics)}")
    print(f"• Saved cleanly to: {output_file}")

else:
    print("\nCould not automatically align ID values.")
    print("Sample Boundary values:")
    print(sa_gdf[boundary_candidates].head())
    print("Sample SAPS CSV values:")
    print(saps_df[saps_candidates].head())

Loading raw files...
Scanning candidate boundary keys: ['OBJECTID', 'SA_GUID_2016', 'SA_GUID_2022', 'SA_PUB2011', 'SA_PUB2016', 'SA_PUB2022', 'SA_GEOGID_2022', 'SA_CHANGE_CODE', 'SA_URBAN_AREA_FLAG', 'SA_URBAN_AREA_NAME', 'SA_NUTS1', 'SA_NUTS1_NAME', 'SA_NUTS2', 'SA_NUTS2_NAME', 'SA_NUTS3', 'SA_NUTS3_NAME', 'ED_GUID', 'ED_ID_STR', 'COUNTY_CODE']
Scanning candidate SAPS CSV keys: ['GUID', 'GEOGID', 'GEOGDESC', 'T1_2WIDM', 'T1_2WIDF', 'T1_2WIDT']

Success! Optimal match found:
• Boundary Key: 'SA_GUID_2022' <---> SAPS Key: 'GUID'
• Matching Small Area IDs: 18919

Reprojecting to EPSG:2157 (ITM meters)...

--- Demand Layer Processing Complete ---
• Total Small Areas in Dundrum-Sandyford study area: 385
• Saved cleanly to: C:\Users\My PC\Desktop\Dublin_Mobility_Project\processed_data\dublin_demographics_sa2022.geojson


# process OSM Active Travel Vector Network

In [7]:
print("Loading dublin_active_travel.geojson (OSM vector network)...")
gdf = gpd.read_file(raw_dir / "dublin_active_travel.geojson")
print(f"• Total features loaded: {len(gdf)}")

# 2. Filter for line geometries (roads/paths), removing point nodes/crossings
lines_gdf = gdf[gdf.geometry.type.isin(['LineString', 'MultiLineString'])].copy()
print(f"• Linear network segments (LineString/MultiLineString): {len(lines_gdf)}")

# 3. Helper function to classify cycling infrastructure from OSM tags
def classify_osm_cycleway(row):
    highway = str(row.get('highway', '')).lower()
    cycleway = str(row.get('cycleway', '')).lower()
    segregated = str(row.get('segregated', '')).lower()
    bicycle = str(row.get('bicycle', '')).lower()
    
    # Off-road / Segregated dedicated cycle paths
    if highway == 'cycleway' or cycleway in ['track', 'opposite_track'] or segregated == 'yes':
        return 'SEPARATE_TRACK'
    # Dedicated on-road cycle lanes
    elif cycleway in ['lane', 'opposite_lane', 'shared_lane']:
        return 'CYCLE_LANE'
    # Shared footways/paths where cycling is permitted
    elif highway in ['path', 'footway', 'pedestrian'] and bicycle in ['yes', 'designated', 'permitted']:
        return 'SHARED_PATH'
    # General road network tagged as accessible for bicycles
    elif bicycle in ['yes', 'designated', 'permitted'] or highway in ['residential', 'tertiary', 'secondary', 'unclassified', 'living_street']:
        return 'SHARED_ROAD'
    else:
        return 'OTHER_PATH'

lines_gdf['BIKE_TYPE'] = lines_gdf.apply(classify_osm_cycleway, axis=1)

# 4. Standardize projection to EPSG:2157 (Irish Transverse Mercator - meters)
if lines_gdf.crs is None:
    lines_gdf = lines_gdf.set_crs(epsg=4326)

print("Reprojecting spatial network to EPSG:2157 (ITM meters)...")
network_itm = lines_gdf.to_crs(epsg=2157)

# 5. Spatial clip to Dundrum - Sandyford Study Area
study_bbox_wgs84 = box(-6.270, 53.250, -6.180, 53.300)
bbox_itm = gpd.GeoSeries([study_bbox_wgs84], crs="EPSG:4326").to_crs(epsg=2157).geometry.iloc[0]

dundrum_network = network_itm[network_itm.intersects(bbox_itm)].copy()

# Calculate segment length in meters
dundrum_network['length_m'] = dundrum_network.geometry.length

# 6. Clean list/dict object attributes for GeoJSON export
for col in dundrum_network.columns:
    if dundrum_network[col].apply(lambda x: isinstance(x, (list, dict))).any():
        dundrum_network[col] = dundrum_network[col].astype(str)

# 7. Save output file
output_path = processed_dir / "processed_dublin_cycle_infrastructure.geojson"
dundrum_network.to_crs(epsg=4326).to_file(output_path, driver="GeoJSON")

print("\n" + "="*60)
print(" ACTIVE TRAVEL NETWORK PROCESSING COMPLETE")
print("="*60)
print(f"• Total linear cycle network segments in corridor: {len(dundrum_network)}")
total_km = dundrum_network['length_m'].sum() / 1000.0
print(f"• Total network length in study area: {total_km:.2f} km")

print("\nInfrastructure Typology Breakdown in Study Area:")
type_summary = dundrum_network.groupby('BIKE_TYPE')['length_m'].sum() / 1000.0
for b_type, length_km in type_summary.items():
    print(f"  - {b_type}: {length_km:.2f} km")

print(f"\nSaved cleanly to:\n {output_path}")

Loading dublin_active_travel.geojson (OSM vector network)...
• Total features loaded: 5564
• Linear network segments (LineString/MultiLineString): 5336
Reprojecting spatial network to EPSG:2157 (ITM meters)...

 ACTIVE TRAVEL NETWORK PROCESSING COMPLETE
• Total linear cycle network segments in corridor: 5336
• Total network length in study area: 430.36 km

Infrastructure Typology Breakdown in Study Area:
  - OTHER_PATH: 363.61 km
  - SEPARATE_TRACK: 62.53 km
  - SHARED_PATH: 4.22 km

Saved cleanly to:
 C:\Users\My PC\Desktop\Dublin_Mobility_Project\processed_data\processed_dublin_cycle_infrastructure.geojson


# Process & integrate protected cycle infrastructure 2025

In [8]:
raw_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "raw"

# Search for all spatial vector formats
vector_files = list(raw_dir.glob("*.geojson")) + list(raw_dir.glob("*.shp")) + list(raw_dir.glob("*.gpkg"))

print(f"Found {len(vector_files)} vector spatial file(s) in data/raw:\n")
for f in vector_files:
    try:
        gdf = gpd.read_file(f)
        print(f"• {f.name}")
        print(f"   - Features: {len(gdf)}")
        print(f"   - Geometry Types: {list(gdf.geometry.type.unique())}")
        print(f"   - Key Columns: {list(gdf.columns[:6])}\n")
    except Exception as e:
        print(f"• {f.name} (Error reading: {e})\n")

Found 3 vector spatial file(s) in data/raw:

• cso_small_area_boundaries.geojson
   - Features: 18919
   - Geometry Types: ['Polygon', 'MultiPolygon']
   - Key Columns: ['OBJECTID', 'SA_GUID_2016', 'SA_GUID_2022', 'SA_PUB2011', 'SA_PUB2016', 'SA_PUB2022']

• dublin_active_travel.geojson
   - Features: 5564
   - Geometry Types: ['Point', 'Polygon', 'LineString']
   - Key Columns: ['element', 'id', 'highway', 'button_operated', 'crossing', 'crossing:island']

• dublin_street_edges.geojson
   - Features: 48825
   - Geometry Types: ['LineString']
   - Key Columns: ['u', 'v', 'key', 'osmid', 'highway', 'lanes']



C:\Users\My PC\anaconda3\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


# process GTFS public transit data (luas stops)

In [9]:
processed_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "processed"

# Load the GTFS or Luas stops layer
stops_path = processed_dir / "processed_gtfs_stops_dundrum.geojson"
stops_gdf = gpd.read_file(stops_path).to_crs(epsg=2157)

# Define our 5 target Luas stations
target_luas = ['Dundrum', 'Balally', 'Kilmacud', 'Stillorgan', 'Sandyford']

# Search stop_name column for the 5 stations
pattern = '|'.join(target_luas)
luas_5_gdf = stops_gdf[stops_gdf['stop_name'].str.contains(pattern, case=False, na=False)].copy()

# Deduplicate in case of multiple platform IDs per station
# Keep one point per station for clean distance buffering
luas_5_gdf['station_clean'] = luas_5_gdf['stop_name'].apply(
    lambda name: next((s for s in target_luas if s.lower() in name.lower()), name)
)
luas_5_clean = luas_5_gdf.drop_duplicates(subset=['station_clean']).copy()

# Save scope-aligned layer
out_luas_path = processed_dir / "processed_luas_5_stations.geojson"
luas_5_clean.to_crs(epsg=4326).to_file(out_luas_path, driver="GeoJSON")

print("="*60)
print(" SCOPE REALIGNMENT: LUAS CORRIDOR STATIONS")
print("="*60)
print(f"• Filtered down to target Luas stops: {len(luas_5_clean)}")
for idx, row in luas_5_clean.iterrows():
    print(f"  - Station: {row['station_clean']} (Stop ID: {row.get('stop_id', 'N/A')})")

print(f"\nSaved cleanly to:\n {out_luas_path.name}")

 SCOPE REALIGNMENT: LUAS CORRIDOR STATIONS
• Filtered down to target Luas stops: 5
  - Station: Kilmacud (Stop ID: 8250DB000444)
  - Station: Stillorgan (Stop ID: 8250DB000447)
  - Station: Dundrum (Stop ID: 8250DB002825)
  - Station: Balally (Stop ID: 8250DB002829)
  - Station: Sandyford (Stop ID: 8250DB003470)

Saved cleanly to:
 processed_luas_5_stations.geojson


## Phase 2 & 3 (Revised. Missed out on some steps): Network Fetching & True Spatial Wrangling
*   Fetching the OpenStreetMap routing network via OSMnx.
*   Filtering out illegal routes (motorways).
*   Projecting to EPSG:2157 (Irish Transverse Mercator).

# a redo to organise the code by loading base datasets and defining spatial boundary

In [10]:
print("Loading base spatial layers...")
# 1. Load Luas 5 Target Stations
luas_gdf = gpd.read_file(processed_dir / "processed_luas_5_stations.geojson").to_crs(epsg=2157)

# 2. Load Census Demographics (Small Areas)
saps_gdf = gpd.read_file(processed_dir / "dublin_demographics_sa2022.geojson").to_crs(epsg=2157)

# 3. Load Dublin Cycle Infrastructure Layer
cycle_gdf = gpd.read_file(processed_dir / "processed_dublin_cycle_infrastructure.geojson").to_crs(epsg=2157)

# 4. Define Spatial Bounding Box (3 km buffer around the 5 Luas stations)
corridor_bounds = luas_gdf.geometry.buffer(3000).total_bounds
corridor_box = box(*corridor_bounds)
corridor_gdf = gpd.GeoDataFrame(geometry=[corridor_box], crs=2157).to_crs(epsg=4326)
bounding_polygon = corridor_gdf.geometry.iloc[0]

print(f"✅ Loaded {len(luas_gdf)} Luas stations, {len(saps_gdf)} demographic areas, and {len(cycle_gdf)} cycle paths.")
print("✅ Dynamic 3km corridor bounding box generated.")

Loading base spatial layers...
✅ Loaded 5 Luas stations, 385 demographic areas, and 5336 cycle paths.
✅ Dynamic 3km corridor bounding box generated.


# fetch OSM street network and filter illegal routes (network filtering)

In [11]:
print("Fetching pedestrian/cycling street network via OSMnx...")

# Fetch network inside the dynamic bounding box
G = ox.graph_from_polygon(bounding_polygon, network_type='all')

# Convert graph to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G)

# Filter out motorways (e.g., M50 slip roads) where walking/cycling is illegal
edges['highway_str'] = edges['highway'].astype(str)
edges_clean = edges[~edges['highway_str'].str.contains('motorway')].copy()

# Reproject to Irish Transverse Mercator (EPSG:2157) for accurate distance math
nodes_2157 = nodes.to_crs(epsg=2157)
edges_2157 = edges_clean.to_crs(epsg=2157)

# Clean up complex object types for GeoJSON export
for col in edges_2157.columns:
    if any(isinstance(val, (list, tuple, dict)) for val in edges_2157[col]):
        edges_2157[col] = edges_2157[col].astype(str)

# Save clean base network
edges_2157.to_file(processed_dir / "osm_corridor_edges.geojson", driver="GeoJSON")
nodes_2157.to_file(processed_dir / "osm_corridor_nodes.geojson", driver="GeoJSON")

print(f"✅ Saved clean street network: {len(edges_2157):,} navigable segments (motorways removed).")

Fetching pedestrian/cycling street network via OSMnx...
✅ Saved clean street network: 87,845 navigable segments (motorways removed).


# attribute tagging
* Testing completion of phase 3
* Clearly breaking down cycle infrastructure in study corridor

In [12]:
print("Tagging OSM street network with Dublin Cycle Infrastructure...")

# 1. Ensuring variables from previous cells are available
if 'edges_2157' not in locals():
    edges_2157 = gpd.read_file(processed_dir / "osm_corridor_edges.geojson")
if 'cycle_gdf' not in locals():
    cycle_gdf = gpd.read_file(processed_dir / "processed_dublin_cycle_infrastructure.geojson").to_crs(epsg=2157)

# 2. Buffer the cycle network by 10 meters to catch slightly misaligned OSM streets
print("Applying 10m spatial buffer to catch misaligned geometries...")
cycle_buffered = cycle_gdf.copy()
cycle_buffered.geometry = cycle_buffered.geometry.buffer(10)

# 3. Keep only the necessary classification column (assuming 'BIKE_TYPE' exists from your earlier work)
# If your column is named differently, update 'BIKE_TYPE' to match your data.
infra_col = 'BIKE_TYPE' if 'BIKE_TYPE' in cycle_buffered.columns else cycle_buffered.columns[0]
cycle_subset = cycle_buffered[[infra_col, 'geometry']]

# 4. Spatial Join: Which OSM edges intersect the buffered cycle paths?
print("Merging infrastructure tags onto street segments...")
tagged_edges = gpd.sjoin(edges_2157, cycle_subset, how='left', predicate='intersects')

# 5. Clean up duplicates (If a street intersects multiple cycle paths, drop the duplicates)
tagged_edges = tagged_edges[~tagged_edges.index.duplicated(keep='first')]

# 6. Fill missing values (Streets that didn't join anything are regular roads/paths)
tagged_edges['cycle_infra'] = tagged_edges[infra_col].fillna('None / Mixed Traffic')

# 7. Drop the temporary index and infrastructure column from the join
if 'index_right' in tagged_edges.columns:
    tagged_edges = tagged_edges.drop(columns=['index_right'])
if infra_col in tagged_edges.columns:
    tagged_edges = tagged_edges.drop(columns=[infra_col])

# 8. Save the fully attributed routing network for Phase 4
out_tagged = processed_dir / "osm_corridor_edges_tagged.geojson"
tagged_edges.to_file(out_tagged, driver="GeoJSON")

print("="*60)
print(f"✅ Phase 3 Complete: Attribute Tagging Successful!")
print(f"📁 Network saved to: {out_tagged.name}")
print("\nInfrastructure Breakdown on Network Edges:")
print(tagged_edges['cycle_infra'].value_counts())
print("="*60)

Tagging OSM street network with Dublin Cycle Infrastructure...
Applying 10m spatial buffer to catch misaligned geometries...
Merging infrastructure tags onto street segments...
✅ Phase 3 Complete: Attribute Tagging Successful!
📁 Network saved to: osm_corridor_edges_tagged.geojson

Infrastructure Breakdown on Network Edges:
cycle_infra
None / Mixed Traffic    45096
OTHER_PATH              38154
SEPARATE_TRACK           4246
SHARED_PATH               349
Name: count, dtype: int64


## Phase 4
* Calculating True Isochrones (Netwowrk based Travel Time Zones
* Generating walking Isochrones for 400m and 800m distances
* Generating cycling Isochrones for 1.5KM and 3KM distances
* Assigning a corridor safety score ranging 0-100 to each street segement

# calculating True Isochrones

In [13]:
import networkx as nx
from shapely.geometry import Point

print("Preparing the network graph for true travel-time routing...")
processed_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "processed"

# 1. Load Luas stations to use as origin nodes
luas_gdf = gpd.read_file(processed_dir / "processed_luas_5_stations.geojson").to_crs(epsg=2157)

# 2. Prepare the graph (Projecting the graph to EPSG:2157 for meter-based distance)
if 'G' in locals():
    # If G is still in memory from Cell 3, project it directly
    G_proj = ox.project_graph(G, to_crs="EPSG:2157")
else:
    raise ValueError("The street graph 'G' is not in memory. Please re-run Cell 3 to fetch the network.")

# Fix for OSMnx v2.0+: Use native networkx to make the graph undirected
G_undir = G_proj.to_undirected()

# 3. Find the nearest network node to each of the 5 Luas stations
print("Snapping Luas stations to the street network...")
X = luas_gdf.geometry.x.tolist()
Y = luas_gdf.geometry.y.tolist()
luas_nodes = ox.distance.nearest_nodes(G_undir, X, Y)

# 4. Function to generate Isochrones (Travel Time Catchments)
def generate_isochrone(graph, center_nodes, distance_m):
    polygons = []
    for node in center_nodes:
        # Calculate a sub-graph of all streets reachable within the distance limit
        subgraph = nx.ego_graph(graph, node, radius=distance_m, distance='length')
        
        # Extract the point coordinates of all reachable nodes
        node_points = [Point((data['x'], data['y'])) for n, data in subgraph.nodes(data=True)]
        
        # Create a bounding polygon (Convex Hull) around the reachable points
        if len(node_points) >= 3:
            # Updated to union_all() to avoid Geopandas deprecation warnings
            bounding_poly = gpd.GeoSeries(node_points).union_all().convex_hull
            polygons.append(bounding_poly)
            
    # Merge overlapping station catchments into one master polygon
    if polygons:
        # Updated to union_all() to avoid Geopandas deprecation warnings
        merged_poly = gpd.GeoSeries(polygons).union_all()
        return gpd.GeoDataFrame(geometry=[merged_poly], crs=2157)
    else:
        # Return an empty GeoDataFrame if no polygons were created
        return gpd.GeoDataFrame(geometry=[], crs=2157)

# 5. Generate the Roadmap-Specified Isochrones
print("Generating 400m (5-min) and 800m (10-min) Walking Catchments...")
walk_400m = generate_isochrone(G_undir, luas_nodes, 400)
walk_800m = generate_isochrone(G_undir, luas_nodes, 800)

print("Generating 1.5km (5-min) and 3km (10-min) Cycling Catchments...")
cycle_1500m = generate_isochrone(G_undir, luas_nodes, 1500)
cycle_3000m = generate_isochrone(G_undir, luas_nodes, 3000)

# 6. Save final true-network boundaries
walk_400m.to_file(processed_dir / "isochrone_walk_400m.geojson", driver="GeoJSON")
walk_800m.to_file(processed_dir / "isochrone_walk_800m.geojson", driver="GeoJSON")
cycle_1500m.to_file(processed_dir / "isochrone_cycle_1500m.geojson", driver="GeoJSON")
cycle_3000m.to_file(processed_dir / "isochrone_cycle_3000m.geojson", driver="GeoJSON")

print("="*65)
print("✅ True Network Isochrones Generated Successfully (Warning Free)!")
print("Files saved to data/processed directory.")
print("="*65)

Preparing the network graph for true travel-time routing...
Snapping Luas stations to the street network...
Generating 400m (5-min) and 800m (10-min) Walking Catchments...
Generating 1.5km (5-min) and 3km (10-min) Cycling Catchments...
✅ True Network Isochrones Generated Successfully (Warning Free)!
Files saved to data/processed directory.


# corridor safety score

In [14]:
# Calculate Corridor Safety Scores (0-100)
processed_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "processed"
edges_gdf = gpd.read_file(processed_dir / "osm_corridor_edges_tagged.geojson")

def calculate_safety_score(row):
    infra = str(row.get('cycle_infra', ''))
    highway = str(row.get('highway', ''))
    
    if infra == 'SEPARATE_TRACK':
        return 100
    elif infra == 'OTHER_PATH':
        return 85
    elif infra == 'SHARED_PATH':
        return 75
    else:
        if 'residential' in highway or 'living_street' in highway or 'pedestrian' in highway:
            return 60
        elif 'tertiary' in highway or 'unclassified' in highway:
            return 40
        elif 'secondary' in highway:
            return 20
        elif 'primary' in highway or 'trunk' in highway:
            return 10
    return 30

edges_gdf['safety_score'] = edges_gdf.apply(calculate_safety_score, axis=1)
out_scored = processed_dir / "osm_corridor_safety_scored.geojson"
edges_gdf.to_file(out_scored, driver="GeoJSON")
print("✅ Corridor Safety Scoring Complete!")

C:\Users\My PC\anaconda3\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


✅ Corridor Safety Scoring Complete!


## Phase 5: GAP Analysis & Business Insights
* Identify high friction zones
* Quantify impact
  

In [15]:
# Gap Analysis & High-Friction Zone Identification
print("Running Phase 5: Gap Analysis & High-Friction Zone Extraction...")
processed_dir = Path.home() / "Desktop" / "dublin_mobility_project" / "data" / "processed"

# 1. Load data layers
edges_scored = gpd.read_file(processed_dir / "osm_corridor_safety_scored.geojson").to_crs(epsg=2157)
saps_gdf = gpd.read_file(processed_dir / "dublin_demographics_sa2022.geojson").to_crs(epsg=2157)
cycle_3km = gpd.read_file(processed_dir / "isochrone_cycle_3000m.geojson").to_crs(epsg=2157)
walk_800m = gpd.read_file(processed_dir / "isochrone_walk_800m.geojson").to_crs(epsg=2157)

# Identify population column dynamically
pop_col = next((col for col in ['Population', 'Total_Pop', 'T1_1AGETM', 'population'] if col in saps_gdf.columns), saps_gdf.columns[0])

# 2. Calculate populations within catchments
saps_centroids = saps_gdf.copy()
saps_centroids.geometry = saps_centroids.geometry.centroid

# Small areas intersecting catchments
sa_in_cycle = saps_gdf[saps_centroids.intersects(cycle_3km.geometry.iloc[0])]
sa_in_walk = saps_gdf[saps_centroids.intersects(walk_800m.geometry.iloc[0])]

total_pop_cycle = sa_in_cycle[pop_col].sum()
total_pop_walk = sa_in_walk[pop_col].sum()

# 3. Identify High-Friction Zones (High population density Small Areas with low average local safety)
# Spatial join Small Areas to the scored edges to find local network safety per neighborhood
print("Isolating residential pockets choked by low-safety corridors...")
saps_buffered = saps_gdf.copy()
saps_buffered.geometry = saps_buffered.geometry.buffer(50) # 50m buffer to capture adjacent streets

joined_sa_edges = gpd.sjoin(edges_scored, saps_buffered, how='inner', predicate='intersects')
sa_safety_summary = joined_sa_edges.groupby('GEOGID' if 'GEOGID' in saps_gdf.columns else saps_gdf.columns[0]).agg({
    'safety_score': 'mean',
    pop_col: 'first'
}).reset_index()

# High-friction zones: High population (above median) but low local safety score (< 50)
median_pop = sa_safety_summary[pop_col].median()
high_friction_zones = sa_safety_summary[
    (sa_safety_summary[pop_col] > median_pop) & 
    (sa_safety_summary['safety_score'] < 50)
]

# 4. Print Executive Metrics
print("="*65)
print(" 📊 PHASE 5 GAP ANALYSIS: QUANTIFIED IMPACT REPORT 📊")
print("="*65)
print(f"👥 Residents within 10-Min Walk (800m):  {total_pop_walk:,.0f}")
print(f"🚲 Residents within 10-Min Cycle (3km): {total_pop_cycle:,.0f}")
print(f"🚨 Identified High-Friction Neighborhoods: {len(high_friction_zones):,} residential zones")
print(f"⚠️  Vulnerable Population in High-Friction Zones: {high_friction_zones[pop_col].sum():,.0f} residents")
print("="*65)

# Save high-friction analysis table
high_friction_zones.to_csv(processed_dir / "phase5_high_friction_neighborhoods.csv", index=False)
print("📁 High-friction neighborhood metrics saved to data/processed.")
print("="*65)

Running Phase 5: Gap Analysis & High-Friction Zone Extraction...


C:\Users\My PC\anaconda3\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Isolating residential pockets choked by low-safety corridors...
 📊 PHASE 5 GAP ANALYSIS: QUANTIFIED IMPACT REPORT 📊
👥 Residents within 10-Min Walk (800m):  12,289
🚲 Residents within 10-Min Cycle (3km): 53,112
🚨 Identified High-Friction Neighborhoods: 12 residential zones
⚠️  Vulnerable Population in High-Friction Zones: 2,063 residents
📁 High-friction neighborhood metrics saved to data/processed.
